# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melikekaya01/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

1. **What does one row mean?**  
   One row in the final feature frame represents one pseudonymized content page at the end of the March 2026 decision window.

2. **Which table will I use?**  
   I will use the `fact_content_daily_performance` table. Its source grain is one row per report date, pseudonymized client, and pseudonymized content page.

3. **Which time window will I use?**  
   March 2026 is the feature-development month. June 2026 remains sealed as the final test month.

4. **What will I predict or rank?**  
   I will rank content pages for human review based on CTR opportunity. The operational proxy is 1 when a page has at least 500 impressions, an average position of 20 or better, and a CTR below 0.5%, and 0 otherwise.

5. **What will I deliberately exclude?**  
   I will exclude client names, URLs, private search queries, pseudonymous identifiers as model features, and any field derived directly from the proxy label.

In [1]:
!pip -q install --upgrade duckdb huggingface_hub

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from IPython.display import display

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN bulunamadı. Colab Secrets bölümüne "
        "HF_TOKEN adıyla Hugging Face Read token ekleyin."
    )

os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect(database=":memory:")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

WAREHOUSE_ROOT = "hf://datasets/FlyRank/internship-warehouse"

FEATURE_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

FEATURE_PATH = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    f"month={FEATURE_MONTH}/*.parquet"
)

OUTCOME_PATH = (
    f"{WAREHOUSE_ROOT}/fact_content_daily_performance/"
    f"month={OUTCOME_MONTH}/*.parquet"
)

print("DuckDB connection created.")
print("Feature month:", FEATURE_MONTH)
print("Outcome month:", OUTCOME_MONTH)
print("June 2026 remains sealed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 16.5 MB/s eta 0:00:00
DuckDB connection created.
Feature month: 2026-03
Outcome month: 2026-04
June 2026 remains sealed.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features

The final feature frame will contain no more than five features:

1. `log_impressions` — the logarithm of total March search impressions.
2. `log_clicks` — the logarithm of total March search clicks.
3. `weighted_avg_position` — the impression-weighted average search position during March.
4. `active_search_days` — the number of March dates on which the page recorded at least one search impression.
5. `ga4_engagement_rate` — the share of sessions that were engaged, calculated only where GA4 data was available.

#### Label / proxy

- `ctr_opportunity_proxy` is the operational proxy for this lane.
- It equals 1 when a page has at least 500 March impressions, an average position of 20 or better, and an observed CTR below 0.5%.
- It equals 0 otherwise.
- In decimal form, a CTR of 0.5% is represented as `0.005`.

This proxy is designed for prioritization. It does not prove that a page has poor content or that changing the page will cause CTR to increase.

#### Context

- `client_hash_id` — used only for grouping, joins, and future client-aware validation.
- `content_hash_id` — used only to combine daily observations belonging to the same pseudonymized page.
- `report_date` — used to define the March observation window.
- `month` — used to select the correct warehouse partition.
- Total impressions, total clicks, and observed CTR may be retained temporarily to construct and audit the proxy.

#### Excluded

- `client_hash_id` and `content_hash_id` are excluded from the model feature list because they are identifiers rather than transferable page signals.
- Client names, domains, URLs, and private queries are excluded for privacy and because they are not required for this decision.
- `ctr_opportunity_proxy` and any direct copy or transformation of it are excluded because they reveal the answer.
- Raw observed CTR is excluded from the honest feature set because CTR is used directly to construct the proxy label.
- Rows with unavailable GA4 measurements are excluded from GA4-derived calculations because zero-filled values must not be interpreted as measured zero engagement.
- June 2026 is excluded from development because it is the final month and should remain sealed for later testing.

In [2]:
schema_df = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
""").df()

display(schema_df)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [3]:
required_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_data_available",
}

available_columns = set(schema_df["column_name"].tolist())
missing_columns = required_columns - available_columns

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

print("All required columns are available.")

All required columns are available.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification plan

The contract is checked with exactly three verification queries.

1. The first query checks whether the declared daily source grain holds.
2. The second query measures the March 2026 row count, page count, client count, and date span.
3. The third query applies `ga4_data_available IS TRUE` and measures how many rows survive the availability filter.

The feature frame and leakage experiment are built after these three verification queries.

### Verification Query 1 — Source grain

The declared source grain is one row per `report_date × client_hash_id × content_hash_id`.

If the query returns zero duplicate groups, the declared daily grain holds for the March 2026 slice.

In [4]:
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_in_group

    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )

    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id

    HAVING COUNT(*) > 1

    ORDER BY rows_in_group DESC
    LIMIT 10
""").df()

display(grain_check)

if grain_check.empty:
    print(
        "Result: zero duplicate groups were found. "
        "The declared daily source grain holds for March 2026."
    )
else:
    print("Warning: duplicate daily groups were found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_in_group


Result: zero duplicate groups were found. The declared daily source grain holds for March 2026.


### Verification Query 2 — Slice size and date span

This query measures the number of observed daily rows, pseudonymized pages, pseudonymized clients, and the actual date range in the March 2026 slice.

In [5]:
slice_summary = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date

    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
""").df()

display(slice_summary)

summary_row = slice_summary.iloc[0]

print(
    f"The March 2026 slice contains "
    f"{int(summary_row['row_count']):,} observed daily rows, "
    f"{int(summary_row['content_count']):,} pseudonymized content pages, "
    f"and {int(summary_row['client_count']):,} pseudonymized clients."
)

print(
    f"The observed date span is "
    f"{summary_row['min_report_date']} to "
    f"{summary_row['max_report_date']}."
)

,row_count,client_count,content_count,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 observed daily rows, 331,437 pseudonymized content pages, and 55 pseudonymized clients.
The observed date span is 2026-03-01 00:00:00 to 2026-03-31 00:00:00.


### Verification Query 3 — GA4 availability

GA4 values may be zero-filled before analytics data becomes available. Therefore, zero does not always mean measured zero engagement.

The query uses `ga4_data_available IS TRUE` and reports how many March rows survive the availability filter.

In [6]:
availability_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS unavailable_rows,

        ROUND(
            100.0
            * COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            )
            / NULLIF(COUNT(*), 0),
            2
        ) AS available_percentage

    FROM read_parquet(
        '{FEATURE_PATH}',
        hive_partitioning = TRUE
    )
""").df()

display(availability_summary)

availability_row = availability_summary.iloc[0]

print(
    f"After applying ga4_data_available IS TRUE, "
    f"{int(availability_row['available_rows']):,} of "
    f"{int(availability_row['total_rows']):,} rows remain "
    f"({availability_row['available_percentage']:.2f}%)."
)

print(
    "Unavailable GA4 rows are not interpreted "
    "as measured zero engagement."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,unavailable_rows,available_percentage
0,9841378,413966,9427412,4.21


After applying ga4_data_available IS TRUE, 413,966 of 9,841,378 rows remain (4.21%).
Unavailable GA4 rows are not interpreted as measured zero engagement.


### Five-feature frame

The feature frame contains one row per pseudonymized content page for March 2026.

#### Feature availability

1. **`log_impressions`**  
   Knowable at the decision moment because it is calculated only from impressions already observed during March.

2. **`log_clicks`**  
   Knowable at the decision moment because it is calculated only from clicks already observed during March.

3. **`weighted_avg_position`**  
   Knowable at the decision moment because it summarizes search positions already measured during March.

4. **`active_search_days`**  
   Knowable at the decision moment because it counts completed March dates with observed search visibility.

5. **`ga4_engagement_rate`**  
   Knowable at the decision moment because it is calculated only from March sessions where `ga4_data_available IS TRUE`.

Observed CTR is used to construct the proxy and is not included in the honest feature list.

In [7]:
feature_frame = con.execute(f"""
    WITH page_summary AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                COALESCE(gsc_impressions, 0)
            ) AS total_impressions,

            SUM(
                COALESCE(gsc_clicks, 0)
            ) AS total_clicks,

            CASE
                WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
                THEN
                    SUM(COALESCE(gsc_clicks, 0)) * 1.0
                    / SUM(COALESCE(gsc_impressions, 0))
                ELSE NULL
            END AS observed_ctr,

            CASE
                WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
                THEN
                    SUM(
                        COALESCE(gsc_avg_position, 0)
                        * COALESCE(gsc_impressions, 0)
                    ) * 1.0
                    / SUM(COALESCE(gsc_impressions, 0))
                ELSE NULL
            END AS weighted_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN COALESCE(gsc_impressions, 0) > 0
                    THEN report_date
                END
            ) AS active_search_days,

            CASE
                WHEN SUM(
                    CASE
                        WHEN ga4_data_available IS TRUE
                        THEN COALESCE(ga4_sessions, 0)
                        ELSE 0
                    END
                ) > 0
                THEN
                    SUM(
                        CASE
                            WHEN ga4_data_available IS TRUE
                            THEN COALESCE(ga4_engaged_sessions, 0)
                            ELSE 0
                        END
                    ) * 1.0
                    /
                    SUM(
                        CASE
                            WHEN ga4_data_available IS TRUE
                            THEN COALESCE(ga4_sessions, 0)
                            ELSE 0
                        END
                    )
                ELSE NULL
            END AS ga4_engagement_rate

        FROM read_parquet(
            '{FEATURE_PATH}',
            hive_partitioning = TRUE
        )

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(COALESCE(gsc_impressions, 0)) > 0
    )

    SELECT
        client_hash_id,
        content_hash_id,

        LN(1 + total_impressions) AS log_impressions,
        LN(1 + total_clicks) AS log_clicks,
        weighted_avg_position,
        active_search_days,
        ga4_engagement_rate,

        total_impressions,
        total_clicks,
        observed_ctr,

        CASE
            WHEN
                total_impressions >= 500
                AND weighted_avg_position <= 20
                AND observed_ctr < 0.005
            THEN 1
            ELSE 0
        END AS ctr_opportunity_proxy,

        '{FEATURE_MONTH}' AS feature_month

    FROM page_summary
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 12)


,client_hash_id,content_hash_id,log_impressions,log_clicks,weighted_avg_position,active_search_days,ga4_engagement_rate,total_impressions,total_clicks,observed_ctr,ctr_opportunity_proxy,feature_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,8.783243,2.079442,6.893301,31,0.0,6523.0,7.0,0.001073,1,2026-03
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,6.118097,0.000000,3.214128,31,NaN,453.0,0.0,0.000000,0,2026-03
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,8.636042,1.945910,6.535346,31,0.0,5630.0,6.0,0.001066,1,2026-03
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,8.506132,2.639057,7.435680,31,0.0,4944.0,13.0,0.002629,1,2026-03
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,3.761200,0.000000,15.785714,21,0.0,42.0,0.0,0.000000,0,2026-03


In [8]:
honest_features = [
    "log_impressions",
    "log_clicks",
    "weighted_avg_position",
    "active_search_days",
    "ga4_engagement_rate",
]

target_column = "ctr_opportunity_proxy"

assert len(honest_features) == 5

missing_features = [
    column
    for column in honest_features
    if column not in feature_frame.columns
]

if missing_features:
    raise ValueError(
        f"Missing feature columns: {missing_features}"
    )

print("Exactly five honest features are defined:")

for number, feature in enumerate(honest_features, start=1):
    print(f"{number}. {feature}")

Exactly five honest features are defined:
1. log_impressions
2. log_clicks
3. weighted_avg_position
4. active_search_days
5. ga4_engagement_rate


In [9]:
quality_summary = pd.DataFrame({
    "column": honest_features + [target_column],

    "missing_count": [
        int(feature_frame[column].isna().sum())
        for column in honest_features + [target_column]
    ],

    "missing_percentage": [
        round(
            100 * feature_frame[column].isna().mean(),
            2
        )
        for column in honest_features + [target_column]
    ],
})

display(quality_summary)

print("CTR opportunity proxy distribution:")

display(
    feature_frame[target_column]
    .value_counts(dropna=False)
    .rename_axis(target_column)
    .reset_index(name="row_count")
)

print(
    "Positive-label rate:",
    round(feature_frame[target_column].mean(), 4)
)

,column,missing_count,missing_percentage
0,log_impressions,0,0.00
1,log_clicks,0,0.00
2,weighted_avg_position,0,0.00
3,active_search_days,0,0.00
4,ga4_engagement_rate,105048,59.44
5,ctr_opportunity_proxy,0,0.00


CTR opportunity proxy distribution:


,ctr_opportunity_proxy,row_count
0,0,135522
1,1,41216


Positive-label rate: 0.2332


### Deliberate leakage experiment

First, I calculate a quick score using only the five approved features.

I then deliberately add `leaked_label_copy`, which directly copies the proxy label. The leaky score is expected to move toward a perfect result because the model is given the answer.

After demonstrating the problem, I delete the leaked column and retain the honest result.

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

experiment_df = feature_frame.copy()

experiment_df = experiment_df[
    experiment_df[target_column].isin([0, 1])
].copy()

if experiment_df[target_column].nunique() < 2:
    raise ValueError(
        "The proxy contains only one class. "
        "AUC cannot be calculated."
    )

def calculate_quick_auc(dataframe, feature_columns, target):
    X = dataframe[feature_columns]
    y = dataframe[target]

    model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            ),
        ]
    )

    model.fit(X, y)
    predicted_probability = model.predict_proba(X)[:, 1]

    return roc_auc_score(y, predicted_probability)

honest_auc = calculate_quick_auc(
    dataframe=experiment_df,
    feature_columns=honest_features,
    target=target_column,
)

print(f"Honest quick AUC: {honest_auc:.4f}")

Honest quick AUC: 0.9737


In [11]:
experiment_df["leaked_label_copy"] = experiment_df[target_column]

leaky_features = honest_features + ["leaked_label_copy"]

leaky_auc = calculate_quick_auc(
    dataframe=experiment_df,
    feature_columns=leaky_features,
    target=target_column,
)

print(f"Honest quick AUC: {honest_auc:.4f}")
print(f"Leaky quick AUC:  {leaky_auc:.4f}")
print(f"Artificial jump:  {leaky_auc - honest_auc:.4f}")

Honest quick AUC: 0.9737
Leaky quick AUC:  1.0000
Artificial jump:  0.0263


In [12]:
experiment_df = experiment_df.drop(
    columns=["leaked_label_copy"]
)

assert "leaked_label_copy" not in experiment_df.columns

print("Leakage column removed successfully.")
print(f"Retained honest quick AUC: {honest_auc:.4f}")
print("Final feature count:", len(honest_features))

Leakage column removed successfully.
Retained honest quick AUC: 0.9737
Final feature count: 5


### Leakage result

The score increased toward a perfect result after `leaked_label_copy` was added.

This improvement is invalid because the leaked field directly revealed the proxy label. I removed the field and retained the score based on the five approved features.

The honest quick score is not a final model-performance claim because this demonstration fits and scores on the same frame. Its purpose is to make leakage visible.

In [13]:
final_feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "feature_month",
        *honest_features,
        target_column,
    ]
].copy()

assert len(honest_features) == 5
assert "total_impressions" not in final_feature_frame.columns
assert "total_clicks" not in final_feature_frame.columns
assert "observed_ctr" not in final_feature_frame.columns
assert "leaked_label_copy" not in final_feature_frame.columns

print("Final frame shape:", final_feature_frame.shape)
print("Final modeling features:", honest_features)

display(final_feature_frame.head())

Final frame shape: (176738, 9)
Final modeling features: ['log_impressions', 'log_clicks', 'weighted_avg_position', 'active_search_days', 'ga4_engagement_rate']


,client_hash_id,content_hash_id,feature_month,log_impressions,log_clicks,weighted_avg_position,active_search_days,ga4_engagement_rate,ctr_opportunity_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03,8.783243,2.079442,6.893301,31,0.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03,6.118097,0.000000,3.214128,31,NaN,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03,8.636042,1.945910,6.535346,31,0.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03,8.506132,2.639057,7.435680,31,0.0,1
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,2026-03,3.761200,0.000000,15.785714,21,0.0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation

A key limitation of this slice is that the warehouse is an unbalanced panel. Clients do not all have the same amount of historical Search Console and Analytics coverage. Therefore, pages from clients with longer measurement histories may be represented more strongly than recently connected clients.

GA4 fields may also be zero-filled before analytics tracking becomes available. These values cannot be interpreted as measured zero engagement unless `ga4_data_available IS TRUE`.

The proxy is based on observed visibility, position, and CTR thresholds. It cannot explain why CTR is low. Low CTR may be related to the title, search intent, SERP layout, brand familiarity, seasonality, query mix, or other factors that are not fully represented in this slice.

The output is directional decision-support for human review. It is not a causal conclusion and does not guarantee that editing a page will improve CTR.

In [14]:
checks = {
    "feature_frame_has_rows": len(final_feature_frame) > 0,

    "exactly_five_features": len(honest_features) == 5,

    "target_exists": (
        target_column in final_feature_frame.columns
    ),

    "target_is_binary": set(
        final_feature_frame[target_column]
        .dropna()
        .unique()
    ).issubset({0, 1}),

    "leak_column_removed": (
        "leaked_label_copy"
        not in final_feature_frame.columns
    ),

    "raw_ctr_excluded": (
        "observed_ctr"
        not in final_feature_frame.columns
    ),

    "ids_not_model_features": (
        "client_hash_id" not in honest_features
        and "content_hash_id" not in honest_features
    ),

    "june_remains_sealed": FEATURE_MONTH != "2026-06",
}

check_df = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values()),
})

display(check_df)

if all(checks.values()):
    print("All automated contract checks passed.")
else:
    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        f"Failed checks: {failed_checks}"
    )


,check,passed
0,feature_frame_has_rows,True
1,exactly_five_features,True
2,target_exists,True
3,target_is_binary,True
4,leak_column_removed,True
5,raw_ctr_excluded,True
6,ids_not_model_features,True
7,june_remains_sealed,True


All automated contract checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.